In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TestApp") \
    .master("local[*]") \
    .getOrCreate()

print(spark.version)

4.1.1


In [6]:
!curl -O https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
 75 67.83M  75 51.53M   0      0 51.49M      0   00:01   00:01         51.53M
100 67.83M 100 67.83M   0      0 55.53M      0   00:01   00:01         51.53M
100 67.83M 100 67.83M   0      0 55.53M      0   00:01   00:01         51.53M
100 67.83M 100 67.83M   0      0 55.53M      0   00:01   00:01         51.53M


In [7]:
df = spark.read.parquet('yellow_tripdata_2025-11.parquet')

In [11]:
df4 = df.repartition(4)

In [12]:
output_path = "yellow_2025_11_output"

df4.write.mode("overwrite").parquet(output_path)

In [15]:
from pyspark.sql.functions import col, to_date

trips_15 = df4.filter(
    to_date(col("tpep_pickup_datetime")) == "2025-11-15"
)

trips_15.count()

162604

In [17]:
from pyspark.sql.functions import col, unix_timestamp, max

df_with_duration = df.withColumn(
    "trip_duration_seconds",
    unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))
)

max_duration_seconds = df_with_duration.agg(max("trip_duration_seconds")).collect()[0][0]
max_duration_hours = max_duration_seconds / 3600

print("Longest trip (hours):", max_duration_hours)

Longest trip (hours): 90.64666666666666


In [18]:
!curl -O https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100  12331 100  12331   0      0  64421      0                              0
100  12331 100  12331   0      0  64402      0                              0
100  12331 100  12331   0      0  64390      0                              0


In [19]:
zone_df = spark.read.option("header", True).csv("taxi_zone_lookup.csv")
zone_df.createOrReplaceTempView("zones")
zone_df.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [22]:
df.createOrReplaceTempView("yellow")

In [23]:
result = spark.sql("""
SELECT z.Zone, COUNT(*) as trips
FROM yellow y
JOIN zones z
  ON y.PULocationID = z.LocationID
GROUP BY z.Zone
ORDER BY trips ASC
LIMIT 1
""")

result.show()

+--------------------+-----+
|                Zone|trips|
+--------------------+-----+
|Governor's Island...|    1|
+--------------------+-----+

